In [7]:
import pandas as pd
from micom.workflows import load_results, GrowthResults
from micom.measures import production_rates
from skbio.diversity import beta_diversity
from itertools import product
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.patches import Rectangle
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

In [2]:
res_simulated = load_results('../goll_et.al_2020_IBS_FMT/models_simulated/res_western_simulated.zip')
res_actual = load_results('../goll_et.al_2020_IBS_FMT/models/res_western.zip')

DonorRecipientMapping = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')[['recipient', 'donor', 'clinical_response']]
DonorRecipientMapping['donor'] = DonorRecipientMapping['donor'].str.split('_').str[0]
DonorRecipientMapping['recipient'] = DonorRecipientMapping['recipient'].str.split('_').str[0].str.replace('-0', '')
DonorRecipientMapping['comparison'] = DonorRecipientMapping.apply(lambda x: x['donor'] + '_vs_' + x['recipient'] + '-0', axis=1)

abundance_actual = (
    res_actual.growth_rates[['sample_id', 'taxon', 'abundance']]
    .query('sample_id.str.contains("D") | sample_id.str.endswith("-6")')
    .assign(
        sample_id=lambda x: np.where(
            ~x['sample_id'].str.startswith('D'),
            x['sample_id'].str.replace('-6', ''),
            x['sample_id']
        )
    )
    .merge(
        DonorRecipientMapping[['recipient', 'donor']],
        left_on='sample_id',
        right_on='recipient',
        how='left'
    )
    .assign(
        comparison=lambda x: np.where(
            x['recipient'].isna(),
            x['sample_id'],  # Donor samples
            x['donor'] + '_vs_' + x['recipient'] + '-0'  # Recipient samples
        )
    )
    .pivot_table(index='taxon', columns='comparison', values='abundance', fill_value=0.0)
)

abundance_actual_donors = abundance_actual.loc[:, ~abundance_actual.columns.str.endswith('-0')]
abundance_actual_recipients = abundance_actual.loc[:, abundance_actual.columns.str.endswith('-0')]

abundance_calculated = (
    res_simulated.growth_rates[['sample_id', 'taxon', 'abundance']]
    .query('~sample_id.isin(@abundance_actual_recipients.columns)')
    .pivot_table(index='taxon', columns='sample_id', values='abundance', fill_value=0.0)
)

abundance_all = pd.concat([abundance_actual_donors, abundance_actual_recipients, abundance_calculated], axis=1).fillna(0)
b = beta_diversity(metric='braycurtis', counts=abundance_all.T.values)
distMatrix = pd.DataFrame(b.data, index=abundance_all.T.index, columns=abundance_all.T.index)
donors = abundance_actual_donors.columns.tolist()
T6Recipients = list(set(abundance_all.columns) - set(abundance_actual_donors.columns))
bray_curtis_dissimilarity = {
    'score': [],
    'donor': [],
    'recipient': []
}

for pairs in product(donors, T6Recipients):
    donor_1 = pairs[0]
    donor_2, recipient = pairs[1].split('_vs_')[0], pairs[1].split('_vs_')[1]
    recipient = recipient.replace('-0', '')
    if donor_1 == donor_2:
        score = distMatrix.loc[pairs[0], pairs[1]]
        bray_curtis_dissimilarity['score'].append(score)
        bray_curtis_dissimilarity['donor'].append(donor_1)
        bray_curtis_dissimilarity['recipient'].append(recipient)

bray_curtis_dissimilarity_table = (
    pd.DataFrame(bray_curtis_dissimilarity)
    .pivot_table(
        index='donor',
        columns='recipient',
        values='score',
        fill_value=0.0
    )
    .transpose()
)

# sort donors and recipients
donor_order = bray_curtis_dissimilarity_table.columns.tolist()
recipient_order = bray_curtis_dissimilarity_table.index.tolist()
bray_curtis_dissimilarity_table = bray_curtis_dissimilarity_table.loc[recipient_order, donor_order]

bray_curtis_dissimilarity = (
    pd.DataFrame(bray_curtis_dissimilarity)
    .merge(DonorRecipientMapping[['recipient', 'clinical_response']], on=['recipient'], how='left')
)
bray_curtis_dissimilarity['clinical_response'] = pd.Categorical(bray_curtis_dissimilarity['clinical_response'], ['responder', 'non-responder'], ordered=True)

# baseline t0
abundance_base = (
    res_actual.growth_rates[['sample_id', 'taxon', 'abundance']]
    .query('sample_id.str.endswith("-0") | sample_id.str.contains("D")')
    .pivot_table(index='taxon', columns='sample_id', values='abundance', fill_value=0.0)
)

b_base = beta_diversity(metric='braycurtis', counts=abundance_base.T.values)
distMatrix_base = pd.DataFrame(b_base.data, index=abundance_base.T.index, columns=abundance_base.T.index)
donors = abundance_base.columns[abundance_base.columns.str.contains('D')].tolist()
recipients = abundance_base.columns[abundance_base.columns.str.endswith('-0')].tolist()
bray_curtis_dissimilarity_base = {
    'score': [],
    'donor': [],
    'recipient': []
}
donors = abundance_base.columns[abundance_base.columns.str.contains('D')].tolist()
recipients = abundance_base.columns[abundance_base.columns.str.endswith('-0')].tolist()
for pairs in product(donors, recipients):
    bray_curtis_dissimilarity_base['score'].append(distMatrix_base.loc[pairs[0], pairs[1]])
    bray_curtis_dissimilarity_base['donor'].append(pairs[0])
    bray_curtis_dissimilarity_base['recipient'].append(pairs[1].replace('-0', ''))

bray_curtis_dissimilarity_base_table = (
    pd.DataFrame(bray_curtis_dissimilarity_base)
    .pivot_table(
        index='donor',
        columns='recipient',
        values='score',
        fill_value=0.0
    )
    .transpose()
)

# sort donors and recipients
bray_curtis_dissimilarity_base_table = bray_curtis_dissimilarity_base_table.loc[recipient_order, donor_order]

delta_bray_curtis = (bray_curtis_dissimilarity_base_table - bray_curtis_dissimilarity_table) / bray_curtis_dissimilarity_base_table # relative change
bray_curtis_dissimilarity_table.to_csv('./data/3C.csv')
bray_curtis_dissimilarity['score'] = 1 - bray_curtis_dissimilarity['score']
bray_curtis_dissimilarity.to_csv('./data/3E.csv')

bray_curtis_dissimilarity = (
    bray_curtis_dissimilarity
    .set_index(['donor', 'recipient'])
    .merge(DonorRecipientMapping[['donor', 'recipient', 'clinical_response']]
           .set_index(['donor', 'recipient']), 
           left_index=True, 
           right_index=True, 
           how='left', 
           suffixes=('', '_y')
    )
    .dropna()
    .reset_index()
    .drop(columns=['clinical_response_y'])
    .assign(score=lambda x: 1 - x['score'])  # convert dissimilarity to similarity
)
bray_curtis_dissimilarity.to_csv('./data/3D.csv')

In [3]:
simulated_res = load_results('../goll_et.al_2020_IBS_FMT/openbiome_simulated_models/res_western_simulated_openbiome.zip')
simulated_growth = simulated_res.growth_rates.copy()
donors = load_results('../goll_et.al_2020_IBS_FMT/openbiome_models/res_western_openbiome.zip').growth_rates['sample_id'].unique()

In [4]:
recipients = simulated_growth['sample_id'].unique()

abundance_openbiome_simulated = (
    simulated_growth[['sample_id', 'taxon', 'abundance']]
    .pivot_table(index='taxon', columns='sample_id', values='abundance', fill_value=0.0)
)

abundance_openbiome_donors = (
    load_results('../goll_et.al_2020_IBS_FMT/openbiome_models/res_western_openbiome.zip')
    .growth_rates[['sample_id', 'taxon', 'abundance']]
    .pivot_table(index='taxon', columns='sample_id', values='abundance', fill_value=0.0)
)

abundance_all_openbiome = pd.concat([abundance_openbiome_donors, abundance_openbiome_simulated], axis=1).fillna(0)
b = beta_diversity(metric='braycurtis', counts=abundance_all_openbiome.T.values)
distMatrix = pd.DataFrame(b.data, index=abundance_all_openbiome.T.index, columns=abundance_all_openbiome.T.index)
bray_curtis_dissimilarity_openbiome = {
    'score': [],
    'donor': [],
    'recipient': []
}

for pairs in product(donors, recipients):
    donor_1 = pairs[0]
    donor_2, recipient = pairs[1].split('_vs_')[0], pairs[1].split('_vs_')[1]
    recipient = recipient.replace('-0', '')
    if donor_1 == donor_2:
        score = distMatrix.loc[pairs[0], pairs[1]]
        bray_curtis_dissimilarity_openbiome['score'].append(score)
        bray_curtis_dissimilarity_openbiome['donor'].append(donor_1)
        bray_curtis_dissimilarity_openbiome['recipient'].append(recipient)

recipients = [r.replace('-0', '') for r in simulated_growth['sample_id'].str.split('_vs_').str[1].unique()]

bray_curtis_dissimilarity_openbiome_table = (
    pd.DataFrame(bray_curtis_dissimilarity_openbiome)
    .pivot_table(
        index='donor',
        columns='recipient',
        values='score',
        fill_value=0.0
    )
    .transpose()
    .loc[recipients, donors]
)

# baseline t0 extracted from original abundance dataframe
abundance_original_recipient_base = (
    abundance_base.loc[:, abundance_base.columns.str.endswith('-0')]
    .assign(
        **{col.replace('-0', ''): abundance_base[col] for col in abundance_base.columns if col.endswith('-0')}
    )
)
abundance_openbiome_base = pd.concat([abundance_openbiome_donors, abundance_original_recipient_base], axis=1).fillna(0)
b_base = beta_diversity(metric='braycurtis', counts=abundance_openbiome_base.T.values)
distMatrix_base = pd.DataFrame(b_base.data, index=abundance_openbiome_base.T.index, columns=abundance_openbiome_base.T.index)
bray_curtis_dissimilarity_openbiome_base = {
    'score': [],
    'donor': [],
    'recipient': []
}
for pairs in product(donors, recipients):
    bray_curtis_dissimilarity_openbiome_base['score'].append(distMatrix_base.loc[pairs[0], pairs[1]])
    bray_curtis_dissimilarity_openbiome_base['donor'].append(pairs[0])
    bray_curtis_dissimilarity_openbiome_base['recipient'].append(pairs[1])

bray_curtis_dissimilarity_openbiome_base_table = (
    pd.DataFrame(bray_curtis_dissimilarity_openbiome_base)
    .pivot_table(
        index='donor',
        columns='recipient',
        values='score',
        fill_value=0.0
    )
    .transpose()
    .loc[recipients, donors]
)

delta_bray_curtis_openbiome = (bray_curtis_dissimilarity_openbiome_base_table - bray_curtis_dissimilarity_openbiome_table) / bray_curtis_dissimilarity_openbiome_base_table # relative change

In [5]:
delta_bray_curtis_combined = pd.concat([delta_bray_curtis, delta_bray_curtis_openbiome], axis=1).dropna(axis=0)
delta_bray_curtis_combined.to_csv('./data/3F.csv')

In [6]:
# pick a recipient, plot all donor engraftment vs scfa flux
scfa = {
    'propionate': 'ppa[e]', 
    'acetate': 'ac[e]', 
    'butyrate': 'but[e]', 
    'Formate': 'for[e]', 
}

gas = {
    'Hydrogen': 'h2[e]', 
    'Methanethiol': 'ch4s[e]'
}

sulfur = {
    'Sulfite': 'so3[e]', 
    'thiosulfate(2-)': 'tsul[e]', 
}

# combine all metabolites to plot
metabolites_to_plot = {**scfa, **gas, **sulfur}

DonorRecipientPairing = {v: k for k, v in zip(DonorRecipientMapping['donor'], DonorRecipientMapping['recipient'])}

simulated_exchanges = simulated_res.exchanges.copy()
simulated_exchanges = (
    simulated_exchanges
    .query('metabolite.isin(@metabolites_to_plot.values())')
    .query('taxon != "medium" and direction == "export"')
    .assign(flux=lambda x: x['flux'] * x['abundance'])
    .groupby(['sample_id', 'metabolite'], as_index=False)
    .agg({'flux': 'sum'})
    .assign(donor=lambda x: x['sample_id'].str.split('_vs_').str[0])
    .assign(recipient=lambda x: x['sample_id'].str.split('_vs_').str[1].str.replace('-0', ''))
    .merge(
        delta_bray_curtis_openbiome.reset_index().melt(id_vars='recipient', var_name='donor', value_name='engraftment rate'),
        on=['donor', 'recipient'],
        how='inner')
    .drop(columns=['sample_id'])
)

# find the calculated exchanges for the actual donors and recipients, replace actual donor recipient pairing with t6 data
exchanges_calculated = (
    res_simulated.exchanges.copy()
    .query('metabolite.isin(@metabolites_to_plot.values())')
    .query('taxon != "medium" and direction == "export"')
    .assign(flux=lambda x: x['flux'] * x['abundance'])
    .groupby(['sample_id', 'metabolite'], as_index=False)
    .agg({'flux': 'sum'})
    .assign(donor=lambda x: x['sample_id'].str.split('_vs_').str[0])
    .assign(recipient=lambda x: x['sample_id'].str.split('_vs_').str[1].str.replace('-0', ''))
    .merge(
        delta_bray_curtis.reset_index().melt(id_vars='recipient', var_name='donor', value_name='engraftment rate'),
        on=['donor', 'recipient'],
        how='inner'
    )
    .drop(columns=['sample_id'])
    .loc[lambda df: ~df.apply(lambda row: DonorRecipientPairing.get(row['recipient']) == row['donor'], axis=1)]
)

exchanges_actual = (
    res_actual.exchanges.copy()
    .query('metabolite.isin(@metabolites_to_plot.values())')
    .query('taxon != "medium" and direction == "export"')
    .query('sample_id.str.endswith("-6")')
    .assign(flux=lambda x: x['flux'] * x['abundance'])
    .groupby(['sample_id', 'metabolite'], as_index=False)
    .agg({'flux': 'sum'})
    .assign(recipient=lambda x: x['sample_id'].str.replace('-6', ''))
    .assign(donor=lambda x: x['recipient'].map(DonorRecipientPairing))
    .drop(columns=['sample_id'])
)

exchanges_combined = pd.concat([exchanges_calculated, exchanges_actual], axis=0)
exchanges_combined_scfa = (
    exchanges_combined
    .query('metabolite.isin(@scfa.values())')
    .groupby(['donor', 'recipient', 'engraftment rate'], as_index=False)
    .agg({'flux': 'sum'})
    .assign(metabolism='scfa metabolism')
)

exchanges_combined_gas = (
    exchanges_combined
    .query('metabolite.isin(@gas.values())')
    .groupby(['donor', 'recipient', 'engraftment rate'], as_index=False)
    .agg({'flux': 'sum'})
    .assign(metabolism='gas metabolism')
)

exchanges_combined_sulfur = (
    exchanges_combined
    .query('metabolite.isin(@sulfur.values())')
    .groupby(['donor', 'recipient', 'engraftment rate'], as_index=False)
    .agg({'flux': 'sum'})
    .assign(metabolism='sulfur metabolism')
)

exchanges_combined_reduced = pd.concat([exchanges_combined_scfa, exchanges_combined_gas, exchanges_combined_sulfur], axis=0)
exchanges_combined_reduced.to_csv('./data/3G.csv')

In [10]:
DonorRecipientMapping = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')
DonorRecipientMapping['donor'] = DonorRecipientMapping['donor'].str.split('_').str[0]
DonorRecipientMapping['recipient'] = DonorRecipientMapping['recipient'].str.split('_').str[0]
res = load_results('../goll_et.al_2020_IBS_FMT/models/res_western.zip')
res_openbiome = load_results('../goll_et.al_2020_IBS_FMT/openbiome_models/res_western_openbiome.zip')
donors = DonorRecipientMapping['donor'].unique().tolist()
recipients_t6 = DonorRecipientMapping['recipient'].str.replace('-0', '-6').tolist()
recipients_t0 = [recipient.replace('-6', '-0') for recipient in recipients_t6]
all_samples = set(donors+recipients_t0+recipients_t6)
growth_rates = res.growth_rates
exchanges = res.exchanges
model_samples = set(growth_rates['sample_id'].unique().tolist())
samples = model_samples.intersection(all_samples)

group_labels = pd.DataFrame(samples, columns=['sample'])
group_labels['type'] = group_labels['sample'].apply(lambda x: 'donor' if x.startswith('D') else 'recipient T0' if x.endswith('-0') else 'recipient T6')
T6_mapping = {row['recipient'].replace('-0', '-6'): row['clinical_response'] for _, row in DonorRecipientMapping.iterrows()}
DonorRecipientMapping_t6 = pd.DataFrame({
    'recipient': list(T6_mapping.keys()),
    'clinical_response': list(T6_mapping.values())
})
group_labels = group_labels.merge(DonorRecipientMapping_t6, left_on='sample', right_on='recipient', how='left')
group_labels['clinical_response'] = group_labels['clinical_response'].fillna(group_labels['type'])
group_labels.set_index('sample', inplace=True)
group_labels = group_labels['clinical_response']

openbiome_labels = res_openbiome.growth_rates[['sample_id']].drop_duplicates().copy()
openbiome_labels['clinical_response'] = 'openbiome donor'
group_labels = pd.concat([group_labels, openbiome_labels.set_index('sample_id')['clinical_response']])

res_production_rates = production_rates(res)[['sample_id', 'metabolite', 'flux', 'name']]
res_production_rates = res_production_rates.pivot_table(
    index='sample_id',
    columns='metabolite',
    values='flux',
    fill_value=0
)

res_production_rates_openbiome = production_rates(res_openbiome)[['sample_id', 'metabolite', 'flux', 'name']]
res_production_rates_openbiome = res_production_rates_openbiome.pivot_table(
    index='sample_id',
    columns='metabolite',
    values='flux',
    fill_value=0
)

def niche_breadth(res_export, threshold=0):
    res_export = res_export.copy()
    res_export = res_export.where(res_export.abs() > threshold, 0)
    presence_absence = (res_export != 0).astype(int)
    df = pd.DataFrame(index=res_export.index, columns=['niche_breadth'])
    df['niche_breadth'] = presence_absence.sum(axis=1)
    return df

niche_breadth_df = niche_breadth(res_production_rates, threshold=1e-2)
niche_breadth_df = niche_breadth_df.merge(group_labels, left_index=True, right_index=True)
niche_breadth_df_openbiome = niche_breadth(res_production_rates_openbiome, threshold=1e-2)
niche_breadth_df_openbiome = niche_breadth_df_openbiome.merge(openbiome_labels.set_index('sample_id'), left_index=True, right_index=True)
niche_breadth_combined = pd.concat([niche_breadth_df, niche_breadth_df_openbiome])
niche_breadth_combined.to_csv('./data/3H.csv')

In [11]:
def niche_effectiveness(res_export, threshold=0):
    res_export = res_export.copy()

    res_export = res_export.transpose() # metabolite x sample
    # compute shannon diversity for each sample
    p = res_export / res_export.sum(axis=0)
    shannon_diversity = -(p * np.log(p.where(p > threshold, 1.0))).sum(axis=0)
    df = pd.DataFrame(index=res_export.columns, columns=['niche_effectiveness'])
    df['niche_effectiveness'] = shannon_diversity
    return df

niche_effectiveness_df = niche_effectiveness(res_production_rates, threshold=1e-2)
niche_effectiveness_df = niche_effectiveness_df.merge(group_labels, left_index=True, right_index=True)
niche_effectiveness_df_openbiome = niche_effectiveness(res_production_rates_openbiome, threshold=1e-2)
niche_effectiveness_df_openbiome = niche_effectiveness_df_openbiome.merge(openbiome_labels.set_index('sample_id'), left_index=True, right_index=True)
niche_effectiveness_combined = pd.concat([niche_effectiveness_df, niche_effectiveness_df_openbiome])
niche_effectiveness_combined.to_csv('./data/3I.csv')

In [12]:
niche_evenness_combined = niche_effectiveness_combined.merge(niche_breadth_combined.drop(columns=['clinical_response']), left_index=True, right_index=True)
niche_evenness_combined['niche_evenness'] = niche_evenness_combined['niche_effectiveness'] / np.log(niche_evenness_combined['niche_breadth'])
niche_evenness_combined.to_csv('./data/3J.csv')